In [10]:
import os
from dotenv import load_dotenv
import openai
from qdrant_client import QdrantClient
from qdrant_client.http import models
import uuid

# Load environment variables
load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")
qdrant = QdrantClient(url=os.getenv("QDRANT_URL"))

COLLECTION_NAME = "documents"
VECTOR_SIZE = 3072  # text-embedding-3-large uses 3072 dimensions

# Create collection (if it doesn't exist yet)
try:
    qdrant.get_collection(COLLECTION_NAME)
except:
    qdrant.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE),
    )

# Function to embed text
def embed_text(text):
    response = openai.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )
    return response.data[0].embedding

# Upload documents from directory
import os

knowledge_dir = "/home/scottschweizer/TradeKnowledge/Knowledge"

for filename in os.listdir(knowledge_dir):
    filepath = os.path.join(knowledge_dir, filename)
    if os.path.isfile(filepath):
        with open(filepath, "r", encoding="utf-8") as file:
            content = file.read()
            embedding = embed_text(content)
            qdrant.upsert(
                collection_name=COLLECTION_NAME,
                points=[
                    models.PointStruct(
                        id=str(uuid.uuid4()),  # Unique ID for each doc
                        vector=embedding,
                        payload={"filename": filename, "text": content}
                    )
                ]
            )
print("✅ Documents uploaded successfully.")


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/home/scottschweizer/TradeKnowledge/Knowledge'